# Cleaning and filtering

This adapted notebook implements a lightweight rule-based language filtering pipeline
for conversational tree-structured data (e.g., ShareGPT-style datasets).

Goal:
- Load raw conversation trees
- Detect English prompts with deterministic heuristics
- Filter non-English trees
- Save the cleaned dataset
- Inspect random sample prompts

Adjustments in this version:
- configuration, processing logic, diagnostics, and execution are separated cleanly
- the stopword heuristic remains the primary decision rule
- Lingua is optional and can run in `support_only` or `strict` mode
- extra diagnostics make heuristic/Lingua disagreements visible


In [1]:
# ----------------------------
# IMPORTS + CONFIG
# ----------------------------

import re
import json
import random
import unicodedata
from pathlib import Path
from time import perf_counter
from lingua import Language, LanguageDetectorBuilder


def find_project_root(start=None, marker="01_data"):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing {marker!r}")


PROJECT_ROOT = find_project_root()

CONFIG = {
    "data_dir_raw": PROJECT_ROOT / "01_data" / "01_raw" / "messy_dataset",
    "data_dir_processed": PROJECT_ROOT / "01_data" / "02_processed",
    "input_files": [
        "sg_90k_part1.json",
        "sg_90k_part2.json",
    ],
    "lingua_language_codes": [
        "ENGLISH",
        "GERMAN",
        "DUTCH",
        "FRENCH",
        "SPANISH",
        "PORTUGUESE",
        "ITALIAN",
        "POLISH",
        "FINNISH",
        "CROATIAN",
        "INDONESIAN",
        "CHINESE",
    ],
    "lingua_min_confidence": 0.75,
    "output_file": "llm_sustainability_raw_v1-0-0.json",
    "sample_size": 10,
    "sample_max_chars": 700,
    "random_seed": 42,
    "min_words": 5,
    "min_english_stopwords": 2,
    "min_margin": 1,
    "min_latin_ratio": 0.8,
    "max_non_ascii_ratio": 0.20,
    "max_digit_ratio": 0.30,
    "lingua_mode": "support_only",
}


In [2]:
# ----------------------------
# STOPWORDS + LINGUA SETUP
# ----------------------------

WORD_RE = re.compile(r"[A-Za-zÀ-ÖØ-öø-ÿ']+")

STOPWORDS = {
    "en": {
        "the","and","to","of","a","in","that","is","it","for","on","with","as","this",
        "be","are","you","your","can","will","not","or","if","we","from","by","at","an",
        "have","has","do","does","what","which","when","where","how","why","about","would",
        "could","should","there","their","they","them","these","those","one","more","use",
        "using","make","write","explain","please","help"
    },
    "de": {
        "der","die","das","und","ist","ich","du","sie","er","es","wir","nicht","ein","eine",
        "zu","mit","auf","für","von","wie","was","warum","wenn","dass","kann","sind","haben",
        "werden","auch","oder","aber","bitte","über","hallo"
    },
    "nl": {
        "de","het","een","en","van","ik","je","niet","dat","dit","die","op","voor","met",
        "als","zijn","hebben","worden","wat","waar","waarom","hoe"
    },
    "vi": {
        "và","là","của","có","cho","trong","một","không","tôi","bạn","hãy","này","để"
    },
    "es": {
        "el","la","los","las","y","de","que","en","un","una","es","por","para","con","como","no","se"
    },
    "fr": {
        "le","la","les","et","de","des","du","un","une","est","dans","pour","avec","que","qui"
    },
    "pt": {
        "o","a","os","as","e","de","que","em","um","uma","é","para","com","como","não"
    },
    "it": {
        "il","lo","la","gli","le","e","di","che","in","un","una","è","per","con"
    },
}

LINGUA_LANGUAGES = [getattr(Language, code) for code in CONFIG["lingua_language_codes"]]
lingua_detector = LanguageDetectorBuilder.from_languages(*LINGUA_LANGUAGES).build()


def detect_language_lingua(text):
    text = (text or "").strip()
    if not text:
        return {
            "language": None,
            "language_code": None,
            "confidence": 0.0,
            "top_confidences": [],
        }

    lang = lingua_detector.detect_language_of(text)
    confidences = lingua_detector.compute_language_confidence_values(text)
    top_confidences = [
        {
            "language": str(item.language),
            "language_code": getattr(item.language.iso_code_639_1, "name", None),
            "confidence": round(item.value, 4),
        }
        for item in confidences[:3]
    ]

    if lang is None:
        return {
            "language": None,
            "language_code": None,
            "confidence": 0.0,
            "top_confidences": top_confidences,
        }

    top_conf = top_confidences[0]["confidence"] if top_confidences else 0.0
    return {
        "language": str(lang),
        "language_code": getattr(lang.iso_code_639_1, "name", None),
        "confidence": top_conf,
        "top_confidences": top_confidences,
    }


In [ ]:
# ----------------------------
# CORE LOGIC
# ----------------------------

def get_first_user_prompt(tree):
    conversations = tree.get("conversations", [])
    for message in conversations:
        if message.get("from") == "human":
            return message.get("value", "")
    return ""


def normalize_words(text):
    return [w.lower().strip("'") for w in WORD_RE.findall(text or "")]


def count_stopwords(words, stopwords_dict):
    return {
        lang: sum(w in stopwords for w in words)
        for lang, stopwords in stopwords_dict.items()
    }


def get_language_scores_from_prompt(prompt):
    words = normalize_words(prompt)
    stopword_scores = count_stopwords(words, STOPWORDS)

    english_score = stopword_scores["en"]
    strongest_other = max(v for k, v in stopword_scores.items() if k != "en")

    return {
        "words": words,
        "word_count": len(words),
        "stopword_scores": stopword_scores,
        "english_score": english_score,
        "strongest_other": strongest_other
    }


def latin_ratio(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0

    latin = 0
    for ch in letters:
        try:
            if "LATIN" in unicodedata.name(ch):
                latin += 1
        except ValueError:
            pass
    return latin / len(letters)


def looks_noisy_or_mixed(text, config):
    words = normalize_words(text)

    if not words:
        return True

    non_ascii_words = sum(not w.isascii() for w in words)
    digit_like = sum(any(ch.isdigit() for ch in w) for w in words)

    non_ascii_ratio = non_ascii_words / len(words)
    digit_ratio = digit_like / len(words)

    return (
        non_ascii_ratio > config["max_non_ascii_ratio"]
        or digit_ratio > config["max_digit_ratio"]
    )


def is_probably_english_lingua(text, min_confidence=0.75):
    result = detect_language_lingua(text)
    return (
        result["language_code"] == "EN"
        and result["confidence"] >= min_confidence
    )


def diagnose_prompt(prompt, config):
    noisy = looks_noisy_or_mixed(prompt, config)
    latin = latin_ratio(prompt)
    scores = get_language_scores_from_prompt(prompt)

    heuristic_core_ok = (
        scores["word_count"] >= config["min_words"]
        and scores["english_score"] >= config["min_english_stopwords"]
        and (scores["english_score"] - scores["strongest_other"]) >= config["min_margin"]
    )

    heuristic_ok = (
        not noisy
        and latin >= config["min_latin_ratio"]
        and heuristic_core_ok
    )

    lingua_result = detect_language_lingua(prompt)
    lingua_ok = (
        lingua_result["language_code"] == "EN"
        and lingua_result["confidence"] >= config["lingua_min_confidence"]
    )

    return {
        "prompt": prompt,
        "word_count": scores["word_count"],
        "english_score": scores["english_score"],
        "strongest_other": scores["strongest_other"],
        "stopword_scores": scores["stopword_scores"],
        "latin_ratio": round(latin, 3),
        "noisy_or_mixed": noisy,
        "heuristic_core_ok": heuristic_core_ok,
        "heuristic_ok": heuristic_ok,
        "lingua_language": lingua_result["language_code"],
        "lingua_confidence": lingua_result["confidence"],
        "lingua_ok": lingua_ok,
        "final_strict_ok": heuristic_ok and lingua_ok,
    }


def is_english_tree(tree, config):
    prompt = get_first_user_prompt(tree)
    scores = get_language_scores_from_prompt(prompt)

    if scores["word_count"] < config["min_words"]:
        return False

    stopword_scores = scores["stopword_scores"]
    english_score = stopword_scores["en"]
    strongest_other = max(v for k, v in stopword_scores.items() if k != "en")

    if not (
        english_score >= config["min_english_stopwords"]
        and (english_score - strongest_other) >= config["min_margin"]
    ):
        return False

    if looks_noisy_or_mixed(prompt, config):
        return False

    return True


def filter_english_trees(data, config):
    return [tree for tree in data if is_english_tree(tree, config)]

In [37]:
# ----------------------------
# IO HELPERS
# ----------------------------

def load_json(path):
    try:
        with Path(path).open("r", encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        print(f"File not found: {path}")
        raise


def load_all_json(paths):
    data = []
    for path in paths:
        data.extend(load_json(path))
    return data


def save_json(data, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2)


In [67]:
# ----------------------------
# DIAGNOSTICS
# ----------------------------

def show_random_questions(data, n=10, max_chars=700, seed=None):
    if seed is not None:
        random.seed(seed)

    sample_size = min(n, len(data))
    for i in random.sample(range(len(data)), sample_size):
        tree = data[i]
        prompt = get_first_user_prompt(tree)
        print("=" * 100)
        print(f"Index: {i}")
        print(prompt[:max_chars])


def show_borderline_cases(data, config, n=30, max_chars=700):
    borderline = []

    for i, tree in enumerate(data):
        prompt = get_first_user_prompt(tree)
        diag = diagnose_prompt(prompt, config)

        margin = diag["english_score"] - diag["strongest_other"]

        if diag["heuristic_ok"]:
            borderline.append({
                "index": i,
                "prompt": prompt,
                "word_count": diag["word_count"],
                "english_score": diag["english_score"],
                "strongest_other": diag["strongest_other"],
                "margin": margin,
                "lingua_code": diag["lingua_language"],
                "lingua_confidence": diag["lingua_confidence"],
            })

    print(f"Borderline cases: {len(borderline)}")

    borderline = sorted(
        borderline,
        key=lambda x: (x["margin"], x["english_score"], x["word_count"])
    )

    for case in borderline[:n]:
        print("=" * 100)
        print(f"Index: {case['index']}")
        print(f"Words: {case['word_count']}")
        print(f"EN score: {case['english_score']}")
        print(f"Strongest other: {case['strongest_other']}")
        print(f"Margin: {case['margin']}")
        print(f"Lingua: {case['lingua_code']} ({case['lingua_confidence']:.4f})")
        print(case["prompt"][:max_chars])


def show_lingua_disagreements(data, config, n=20, max_chars=700):
    disagreements = []

    for i, tree in enumerate(data):
        prompt = get_first_user_prompt(tree)
        diag = diagnose_prompt(prompt, config)

        if diag["heuristic_ok"] and not diag["lingua_ok"]:
            disagreements.append({
                "index": i,
                "prompt": prompt,
                "latin_ratio": diag["latin_ratio"],
                "english_score": diag["english_score"],
                "strongest_other": diag["strongest_other"],
                "lingua_language": diag["lingua_language"],
                "lingua_confidence": diag["lingua_confidence"],
            })

    print(f"Disagreements: {len(disagreements)}")

    disagreements = sorted(
        disagreements,
        key=lambda x: (x["lingua_confidence"], -x["english_score"], x["latin_ratio"])
    )

    for case in disagreements[:n]:
        print("=" * 100)
        print(f"Index: {case['index']}")
        print(f"Latin ratio: {case['latin_ratio']}")
        print(f"EN score: {case['english_score']}")
        print(f"Strongest other: {case['strongest_other']}")
        print(f"Lingua: {case['lingua_language']} ({case['lingua_confidence']:.4f})")
        print(case["prompt"][:max_chars])

    print(f"Lingua disagreements: {len(disagreements)}")


In [68]:
# ----------------------------
# MINI TESTS
# ----------------------------

example_en = {
    "conversations": [
        {"from": "human", "value": "How can I improve my Python code for data analysis?"},
        {"from": "assistant", "value": "You can start by..."}
    ]
}

example_de = {
    "conversations": [
        {"from": "human", "value": "Wie kann ich meinen Python Code verbessern?"},
        {"from": "assistant", "value": "Du kannst..."}
    ]
}

example_short = {
    "conversations": [
        {"from": "human", "value": "Hello there"},
    ]
}

assert get_first_user_prompt(example_en) == "How can I improve my Python code for data analysis?"
assert normalize_words("Hello, World!") == ["hello", "world"]

scores = count_stopwords(["how", "can", "i", "improve", "my", "code"], STOPWORDS)
assert scores["en"] >= 2

result_en = is_english_tree(example_en, CONFIG)
result_de = is_english_tree(example_de, CONFIG)
result_short = is_english_tree(example_short, CONFIG)

print("EN:", result_en)
print("DE:", result_de)
print("SHORT:", result_short)
print(diagnose_prompt(example_en["conversations"][0]["value"], CONFIG))
print(detect_language_lingua("How can I improve my Python code for data analysis?"))
print(detect_language_lingua("czy znasz narzędzie do monitorowania infrastruktury IT - Splunk?"))
print(detect_language_lingua("帮我写一个利用convlstm进行时间序列图像的目标检测"))

assert result_en is True
assert result_de is False
assert result_short is False

print("All mini-tests passed.")


EN: True
DE: False
SHORT: False
{'prompt': 'How can I improve my Python code for data analysis?', 'word_count': 10, 'english_score': 3, 'strongest_other': 0, 'stopword_scores': {'en': 3, 'de': 0, 'nl': 0, 'vi': 0, 'es': 0, 'fr': 0, 'pt': 0, 'it': 0}, 'latin_ratio': 1.0, 'noisy_or_mixed': False, 'heuristic_core_ok': True, 'heuristic_ok': True, 'lingua_language': 'EN', 'lingua_confidence': 0.5386, 'lingua_ok': False, 'final_strict_ok': False}
{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386, 'top_confidences': [{'language': 'Language.ENGLISH', 'language_code': 'EN', 'confidence': 0.5386}, {'language': 'Language.GERMAN', 'language_code': 'DE', 'confidence': 0.1118}, {'language': 'Language.DUTCH', 'language_code': 'NL', 'confidence': 0.0726}]}
{'language': 'Language.POLISH', 'language_code': 'PL', 'confidence': 0.9996, 'top_confidences': [{'language': 'Language.POLISH', 'language_code': 'PL', 'confidence': 0.9996}, {'language': 'Language.GERMAN', 'language_code'

In [69]:
# ----------------------------
# PATHS
# ----------------------------

input_paths = [
    CONFIG["data_dir_raw"] / filename
    for filename in CONFIG["input_files"]
]

output_path = CONFIG["data_dir_processed"] / CONFIG["output_file"]

print("Input paths:")
for path in input_paths:
    print("-", path)

print()
print("Output path:")
print("-", output_path)

Input paths:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part1.json
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\01_raw\messy_dataset\sg_90k_part2.json

Output path:
- c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\02_processed\llm_sustainability_raw_v1-0-0.json


In [70]:
# ----------------------------
# LOAD / SAVE DATA
# ----------------------------

start = perf_counter()

data = load_all_json(input_paths)
english_data = filter_english_trees(data, CONFIG)

duration = perf_counter() - start

print(f"Original: {len(data)}")
print(f"English only: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Time: {duration:.2f}s")
print(f"Trees/sec: {len(data)/duration:.0f}")


Original: 90665
English only: 58863
Removed: 31802
Time: 17.44s
Trees/sec: 5200


In [71]:
# ----------------------------
# RANDOM QUESTIONS
# ----------------------------

show_random_questions(
    english_data,
    n=CONFIG["sample_size"],
    max_chars=CONFIG["sample_max_chars"],
    seed=CONFIG["random_seed"],
)


Index: 41905
I am a 3rd year economics student and one of the courses in the degree is academic literacy.
As part of the course I am required to write a "proposal for a policy paper". The research question I chose is:
"What is the effect of interest rate changes in the economy on the rent in Israel?"

As part of the proposal for the policy paper please write:
1. Importance: The importance of the policy question I have chosen
2. Literature: Literature and relevant findings exist. Concentrate on at least 3 existing articles or reports, with a long and detailed description of each, and an explanation of why it is relevant to the policy question I have chosen
Index: 7296
Act as an interviewer. You ask questions and I answer. Don’t write “Sure, I can do that”.  Address with personal pronouns. Your task is to find out what people think of Netflix. Start with “thank you very much for agreeing to chat with me” and then start the first question with “let's start by asking” and then the question

In [72]:
# ----------------------------
# BORDERLINE CASES
# ----------------------------

show_borderline_cases(
    english_data,
    config=CONFIG,
    n=30,
    max_chars=300,
)


Borderline cases: 57555
Index: 101
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: DE (0.3162)
write johnson's algorithm in python
Index: 698
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.7761)
SUBTOTAL CONSOLIDATION EXPLAIN IN EXCEL
Index: 742
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.3152)
golf in the dominican republic
Index: 2546
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.8700)
Explain DST in simple terms
Index: 2859
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.7192)
write a software project proposal
Index: 4558
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.6827)
applications of IoT in logistics
Index: 4690
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.2925)
name this as BDA_unit 1
Index: 4770
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Lingua: EN (0.7609)
What happened in Triana Alabama?
Index: 5274
Words: 5
EN score: 2
Strongest other: 1
Margin: 1
Li

In [73]:
# ----------------------------
# HEURISTIC / LINGUA DISAGREEMENTS
# ----------------------------

show_lingua_disagreements(
    english_data,
    config=CONFIG,
    n=20,
    max_chars=300,
)


Disagreements: 8721
Index: 11820
Latin ratio: 1.0
EN score: 3
Strongest other: 1
Lingua: DE (0.1376)
Can you scan a youtube video?
Index: 22639
Latin ratio: 1.0
EN score: 3
Strongest other: 1
Lingua: DE (0.1376)
Can you scan a youtube video?
Index: 1123
Latin ratio: 1.0
EN score: 3
Strongest other: 0
Lingua: EN (0.1504)
Write an Blog on SEO
Index: 36847
Latin ratio: 1.0
EN score: 2
Strongest other: 0
Lingua: DE (0.1544)
turn of debug mode on ios codebase
Index: 44686
Latin ratio: 1.0
EN score: 2
Strongest other: 1
Lingua: EN (0.1544)
lim as x-&gt;2 of (x^3-8) / (x-2)
Index: 58402
Latin ratio: 1.0
EN score: 3
Strongest other: 1
Lingua: DE (0.1548)
is a 12 volt system ac or dc
Index: 9228
Latin ratio: 1.0
EN score: 3
Strongest other: 1
Lingua: EN (0.1559)
it is just a test
Index: 56955
Latin ratio: 1.0
EN score: 4
Strongest other: 1
Lingua: IT (0.1577)
How to Pass a django model to vue.js
Index: 53855
Latin ratio: 1.0
EN score: 3
Strongest other: 1
Lingua: EN (0.1613)
Write a DMA in Syst

In [78]:
# ----------------------------
# OPTIONAL CONFIG COMPARISON
# ----------------------------

configs_to_compare = [
    {
        "name": "baseline",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 1,
        "min_latin_ratio": 0.0,
        "max_non_ascii_ratio": 1.0,
        "max_digit_ratio": 1.0,
        "use_lingua": False,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "higher_margin",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.0,
        "max_non_ascii_ratio": 1.0,
        "max_digit_ratio": 1.0,
        "use_lingua": False,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "support_only_lingua",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 0.25,
        "max_digit_ratio": 0.15,
        "lingua_mode": "support_only",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
    {
        "name": "strict_lingua",
        "min_words": 5,
        "min_english_stopwords": 2,
        "min_margin": 2,
        "min_latin_ratio": 0.8,
        "max_non_ascii_ratio": 0.25,
        "max_digit_ratio": 0.15,
        "lingua_mode": "strict",
        "lingua_min_confidence": CONFIG["lingua_min_confidence"],
    },
]

for cfg in configs_to_compare:
    merged_cfg = {**CONFIG, **cfg}
    filtered = filter_english_trees(data, merged_cfg)
    kept_ratio = len(filtered) / len(data)
    print("=" * 80)
    print(merged_cfg["name"])
    print(f"English only: {len(filtered)}")
    print(f"Removed: {len(data) - len(filtered)}")
    print(f"Kept ratio: {kept_ratio:.3f}")


baseline
English only: 58921
Removed: 31744
Kept ratio: 0.650
higher_margin
English only: 57276
Removed: 33389
Kept ratio: 0.632
support_only_lingua
English only: 57254
Removed: 33411
Kept ratio: 0.631
strict_lingua
English only: 57254
Removed: 33411
Kept ratio: 0.631


## Decision on filtering

Both heuristic filtering and Lingua are used, but strict_lingua was selected because it required agreement between the two methods and therefore better removed mixed or noisy prompts. This trade-off favored cleaner, more structurally usable English data over a higher retention rate.

I set lingua_mode to "strict" so that I can continue working with this cleaner final dataset.

In [ ]:
# ----------------------------
# FINAL FILTER CHOICE
# ----------------------------

FINAL_CONFIG = CONFIG.copy()
FINAL_CONFIG.update({
    "min_words": 5,
    "min_english_stopwords": 2,
    "min_margin": 2,
    "min_latin_ratio": 0.8,
    "max_non_ascii_ratio": 0.25,
    "max_digit_ratio": 0.15,
    "lingua_mode": "strict",
    "lingua_min_confidence": 0.75,
})

english_data = filter_english_trees(data, FINAL_CONFIG, STOPWORDS)

print(f"Final English-only dataset: {len(english_data)}")
print(f"Removed: {len(data) - len(english_data)}")
print(f"Kept ratio: {len(english_data) / len(data):.3f}")

Final English-only dataset: 47961
Removed: 42704
Kept ratio: 0.529


In [ ]:
# ----------------------------
# SAVE
# ----------------------------

output_path = FINAL_CONFIG["data_dir_processed"] / FINAL_CONFIG["output_file"]
save_json(english_data, output_path)
